# Development-only model comparison

This notebook is the human-readable analysis layer for the aggregate logistic-regression and Extra Trees development report. The reusable implementation remains in `training/`; this notebook does not duplicate model logic.

Scope and safeguards:

- 500-participant development partition only;
- two candidates, each fitting three separate domain classifiers with the same folds and feature contract;
- no participant identifiers, raw rows, or row-level predictions are displayed;
- the locked 125-participant final-test partition is not evaluated; and
- results are non-diagnostic development evidence, not final clinical performance.

> The operational predicted-probability threshold of 0.50 was reviewed and approved by the consultant pediatrician for this model version.

That approval is recorded only for `logistic-l2-balanced-v1`. Extra Trees uses `0.50` as an experimental comparison cutoff; the numerically identical value does not inherit approval.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display


def find_repository_root() -> Path:
    current = Path.cwd().resolve()
    for candidate_path in (current, *current.parents):
        if (candidate_path / 'pyproject.toml').is_file() and (candidate_path / 'training').is_dir():
            return candidate_path
    raise RuntimeError('Run this notebook from inside the Research_Component repository.')


ROOT = find_repository_root()
REPORT_PATH = ROOT / 'reports' / 'generated' / 'development-model-comparison-report.json'
if not REPORT_PATH.is_file():
    raise FileNotFoundError(
        'Aggregate report not found. Run: python -B -m training.experiments.run_development_experiment'
    )

report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
candidate_labels = {
    'logistic_l2_balanced': 'Logistic regression',
    'extra_trees_balanced_leaf5': 'Extra Trees',
}
candidates = report['candidates']
comparison_threshold = report['comparison_threshold']
logistic_threshold = candidates['logistic_l2_balanced']['decision_threshold']
extra_trees_threshold = candidates['extra_trees_balanced_leaf5']['decision_threshold']

assert report['scope'] == 'development_only'
assert report['final_test_evaluated'] is False
assert set(candidates) == set(candidate_labels)
assert comparison_threshold['value'] == 0.50
assert logistic_threshold['status'] == 'consultant_pediatrician_reviewed_and_approved'
assert logistic_threshold['version'] == 'domain-probability-threshold-v1'
assert extra_trees_threshold['status'] == 'experimental_not_approved_for_model'
assert extra_trees_threshold['version'] is None

display(Markdown(f"**Loaded:** `{REPORT_PATH.relative_to(ROOT)}`"))

## Target and decision thresholds are different

- The observed follow-up domain score `<= 50` defines the training target.
- A candidate probability `>= 0.50` defines its predicted class in this comparison.
- The logistic `0.50` cutoff is recorded as reviewed and approved for `logistic-l2-balanced-v1`.
- The Extra Trees `0.50` cutoff is experimental and not yet approved for that model configuration.

A score of 50 is not a 50% model probability. They operate at different stages.

In [ ]:
provenance = pd.DataFrame(
    [
        ('Experiment status', report['status']),
        ('Evaluation scope', report['scope']),
        ('Development participants', report['development_participants']),
        ('Final test evaluated', report['final_test_evaluated']),
        ('Candidate configurations', ', '.join(
            candidate['model_configuration_version'] for candidate in candidates.values()
        )),
        ('Feature schema', report['feature_schema_version']),
        ('Target definition', report['target_definition_version']),
        ('Shared comparison threshold', comparison_threshold['value']),
        ('Logistic threshold status', logistic_threshold['status']),
        ('Extra Trees threshold status', extra_trees_threshold['status']),
        ('CV folds × repeats', f"{report['cross_validation']['n_splits']} × {report['cross_validation']['n_repeats']}"),
    ],
    columns=['Field', 'Value'],
)
display(provenance.style.hide(axis='index'))

In [ ]:
def nested_keys(value):
    if isinstance(value, dict):
        for key, child in value.items():
            yield key
            yield from nested_keys(child)
    elif isinstance(value, list):
        for child in value:
            yield from nested_keys(child)


raw_identifier_fields = {'participant_number', 'participant_key', 'baseline_assessment_id'}
present_identifier_fields = raw_identifier_fields.intersection(nested_keys(report))
assert not present_identifier_fields, f'Unsafe report fields: {sorted(present_identifier_fields)}'
display(Markdown('**Privacy check passed:** no raw identifier fields are present in the report.'))

## Side-by-side development performance

These values use each participant's out-of-fold probabilities averaged across the five repeats. PR-AUC is emphasized because positive outcomes are uncommon. The candidates use identical participant folds, which makes their development comparison paired. These are not final-test estimates.

In [ ]:
domain_labels = {
    'target_communication_concern': 'Communication',
    'target_cognitive_concern': 'Cognitive',
    'target_social_emotional_concern': 'Social-emotional',
}
performance_rows = []
for candidate_name, candidate in candidates.items():
    for target_name, domain in candidate['domains'].items():
        metrics = domain['averaged_participant_oof_metrics']
        performance_rows.append(
            {
                'Model': candidate_labels[candidate_name],
                'Domain': domain_labels[target_name],
                'Threshold status': candidate['decision_threshold']['status'],
                'Positive count': domain['positive_count'],
                'Sensitivity': metrics['sensitivity'],
                'Specificity': metrics['specificity'],
                'Precision / PPV': metrics['precision'],
                'NPV': metrics['negative_predictive_value'],
                'F1': metrics['f1'],
                'Balanced accuracy': metrics['balanced_accuracy'],
                'PR-AUC': metrics['average_precision_pr_auc'],
                'ROC-AUC': metrics['roc_auc'],
                'Brier score': metrics['brier_score'],
            }
        )

performance = pd.DataFrame(performance_rows).set_index(['Model', 'Domain'])
numeric_columns = [
    column for column in performance.columns if column not in {'Threshold status'}
]
display(performance.style.format('{:.4f}', subset=numeric_columns))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7), constrained_layout=True)
for row, (candidate_name, candidate) in enumerate(candidates.items()):
    for axis, (target_name, domain) in zip(axes[row], candidate['domains'].items()):
        counts = domain['averaged_participant_oof_metrics']['confusion_matrix']
        matrix = [
            [counts['true_negative'], counts['false_positive']],
            [counts['false_negative'], counts['true_positive']],
        ]
        axis.imshow(matrix, cmap='Blues')
        for matrix_row in range(2):
            for column in range(2):
                axis.text(column, matrix_row, matrix[matrix_row][column], ha='center', va='center')
        axis.set_xticks([0, 1], labels=['Predicted 0', 'Predicted 1'])
        axis.set_yticks([0, 1], labels=['Observed 0', 'Observed 1'])
        axis.set_title(f"{candidate_labels[candidate_name]} — {domain_labels[target_name]}")
fig.suptitle('Development out-of-fold confusion matrices at numeric cutoff 0.50')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharex=True, sharey=True, constrained_layout=True)
for row, (candidate_name, candidate) in enumerate(candidates.items()):
    for axis, (target_name, domain) in zip(axes[row], candidate['domains'].items()):
        bins = pd.DataFrame(domain['calibration_bins'])
        axis.plot([0, 1], [0, 1], '--', color='grey', label='Ideal')
        axis.plot(
            bins['mean_predicted_probability'],
            bins['observed_positive_rate'],
            marker='o',
            label='Observed',
        )
        axis.set_title(f"{candidate_labels[candidate_name]} — {domain_labels[target_name]}")
        axis.set_xlabel('Mean predicted probability')
        axis.grid(alpha=0.25)
axes[0, 0].set_ylabel('Observed positive rate')
axes[1, 0].set_ylabel('Observed positive rate')
axes[0, -1].legend()
fig.suptitle('Aggregate development calibration by candidate')
plt.show()

## Candidate-appropriate explainability

Logistic regression is summarized using standardized coefficient magnitude, direction and fold stability. Extra Trees is summarized using permutation importance calculated only on each held-out validation fold and scored with average precision. Neither explanation establishes causation or clinical importance.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10), constrained_layout=True)
for row, (candidate_name, candidate) in enumerate(candidates.items()):
    for axis, (target_name, domain) in zip(axes[row], candidate['domains'].items()):
        explanation = domain['explanation']
        values = pd.DataFrame(explanation['values'])
        if explanation['method'] == 'standardized_logistic_coefficient_stability':
            values['absolute_mean'] = values['mean_coefficient'].abs()
            selected = values.nlargest(10, 'absolute_mean').sort_values('mean_coefficient')
            axis.barh(
                selected['transformed_feature'],
                selected['mean_coefficient'],
                xerr=selected['standard_deviation'],
                alpha=0.8,
            )
            axis.axvline(0, color='black', linewidth=0.8)
            axis.set_xlabel('Mean coefficient ± fold SD')
        else:
            selected = values.nlargest(10, 'mean_importance').sort_values('mean_importance')
            axis.barh(
                selected['feature'],
                selected['mean_importance'],
                xerr=selected['standard_deviation'],
                alpha=0.8,
            )
            axis.axvline(0, color='black', linewidth=0.8)
            axis.set_xlabel('Held-out PR-AUC importance ± SD')
        axis.set_title(f"{candidate_labels[candidate_name]} — {domain_labels[target_name]}")
fig.suptitle('Model explanations across development folds')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True, constrained_layout=True)
for axis, (candidate_name, candidate) in zip(axes, candidates.items()):
    fold_pr_auc = []
    labels = []
    for target_name, domain in candidate['domains'].items():
        labels.append(domain_labels[target_name])
        fold_pr_auc.append(
            [row['metrics']['average_precision_pr_auc'] for row in domain['fold_results']]
        )
    axis.boxplot(fold_pr_auc, tick_labels=labels, showmeans=True)
    axis.set_title(candidate_labels[candidate_name])
    axis.set_ylabel('Fold PR-AUC')
    axis.grid(axis='y', alpha=0.25)
fig.suptitle('Variation across 25 shared development folds per domain')
plt.show()

## Optional in-memory recomputation

The notebook normally reads the reproducible aggregate JSON. To demonstrate the full computation without writing another report, set the flag below to `True`. The existing modules still enforce the frozen development partition and final-test boundary.

In [ ]:
RUN_DEVELOPMENT_EXPERIMENT_IN_MEMORY = False

if RUN_DEVELOPMENT_EXPERIMENT_IN_MEMORY:
    from training.data.development_data import load_development_dataset
    from training.experiments.evaluation import run_development_cross_validation

    development = load_development_dataset()
    in_memory_results = run_development_cross_validation(development.records)
    print(
        'Recomputed',
        in_memory_results['development_participants'],
        'development participants; no report file was written.',
    )
else:
    print('Using the existing aggregate report; no model was recomputed.')

## Interpretation boundary and next decision

This notebook supports paired development-model review. It does not select a final model, validate Extra Trees' operational threshold, or authorize final-test evaluation. Candidate selection should consider PR-AUC, sensitivity/specificity, calibration, uncertainty and explanation stability across all three domains rather than choosing from one favourable metric. The selected configuration and any calibration must be frozen before the final-test gate is considered.